In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import numpy as np

from IATheory.process_config import init_cosmology, init_grids, init_pt_calculators, compute_kernels_spec, init_lightcone, init_photometric, update_global_config, build_specific_config
from IATheory.compute_observables import model_2p_corr
from IATheory.read_data import read_data_mice, read_data_flamingo

<h3> I define the global configuration </h3>

In [ ]:
config_global = dict(
                    H0 = 68.1,
                    Om_m = 0.306, # Omega matter
                    Om_b = 0.0486, # Omega baryons
                    sigma8 = 0.807,
                    n_s = 0.967,
                    num_k = 10001,
                    rp_model_min = 7.391, # Minimum transverse distance to model in Mpc
                    rp_model_max = 128.016, # Maximum transverse distance to model in Mpc
                    bins_rp_model = 16, # Number of transverse distance bins
                    log10kmin = -5, # minimum k
                    log10kmax = 2, # maximum k
                    l_min = 0, # Minimum l
                    l_max = 10001, # Maximum l
                    steps_l = 10, # Steps in l
                    IA_model = 'TATT', # Model for IA
                    min_scale_cut = 5, # Minimum scale cut to apply in the correlation function in Mpc/h (in case you want to run chains)
                    max_scale_cut = 100, # Maximum scale cut to apply in the correlation function in Mpc/h (in case you want to run chains)
)

<h3> I update the config setup </h3>

In [ ]:
# We need to perform some initial computations from the global configuration. This includes definining the CCL cosmo library, define k and rp arrays and initialise PT calculators.
config_global = update_global_config(config_global)

In [ ]:
# I consider the case of lightcone_spec
case = 'lightcone_spec'
config_setup_lightcone_spec = dict(z_min = 0., # Minimum redshift to model
                                   z_max = 1.1, # Maximum redshift to model
                                   z_type = 'spec', # It can either be "phot" or "spec"
                                  )
config_specific = build_specific_config(config_global, config_setup_lightcone_spec, case)

<h3> I read the data </h3>

In [ ]:
def initialize_data():
    
    if case != 'box':
        rp_data, corr, cov_mat = read_data_mice.read_data_mice(config_specific)
    else:
        rp_data, corr, cov_mat = read_data_flamingo.read_data_flamingo(config_specific)

    return rp_data, corr, cov_mat

In [ ]:
rp_data, corr, cov_mat = initialize_data()

<h3> I define the prior, likelihood, and probability functions </h3>

In [ ]:
def log_prior(p):
    
    b_1 = p[0]
    b_2 = p[1]
    if config_specific['IA_model'] == 'NLA':
        a_1 = p[2]
        if not ((0 < b_1 < 2) & (-8 < a_1 < 8)):
            return -np.inf
    
    elif config_specific['IA_model'] == 'TATT':
        a_1 = p[2]
        a_2 = p[3]
        a_d = p[4]
        if not ((0 < b_1 < 2) & (-8 < a_1 < 8) & (-12 < a_2 < 12) & (-12 < a_d < 12)):
            return -np.inf
    
    mu = 0
    sigma = 0.5
    return np.log(1.0/(np.sqrt(2*np.pi)*sigma))-0.5*(b_2-mu)**2/sigma**2

def interpolate_model(model_wgg, model_wgp):

    corr_model_wgg_interpol = np.interp(rp_data, config_specific['rp_model'], model_wgg)
    corr_model_wgp_interpol = np.interp(rp_data, config_specific['rp_model'], model_wgp)
    corr_model_interpol = np.concatenate([corr_model_wgg_interpol, corr_model_wgp_interpol])
    return corr_model_interpol

def log_likelihood(p):

    galaxy_bias = p[0:2]
    ia_params = p[2:5]

    model = model_2p_corr(config_global, galaxy_bias, ia_params)

    if case == 'box':
        model.model_wgg_spec_snapshot(config_specific)
        model.model_wgp_spec_snapshot(config_specific)
        corr_model_interpol = interpolate_model(model.wgg_spec_snapshot.xi, model.wgp_spec_snapshot.xi)
    elif case == 'lightcone_spec':
        model.model_wgg_spec_lightcone(config_specific)
        model.model_wgp_spec_lightcone(config_specific)
        corr_model_interpol = interpolate_model(model.wgg_spec_lightcone.xi, model.wgp_spec_lightcone.xi)
    elif case == 'lightcone_phot':
        model.model_wgg_phot_lightcone(config_specific)
        model.model_wgp_phot_lightcone(config_specific)
        corr_model_interpol = interpolate_model(model.wgg_phot_lightcone.xi, model.wgp_phot_lightcone.xi)
    else:
        print('Incorrect case')

    delta = (corr - corr_model_interpol)
    inv_cov = np.linalg.pinv(cov_mat)
    chisq = delta.dot(inv_cov.dot(delta.T))
    ll = -0.5 * chisq
    if np.isnan(ll):
        return -np.inf, chisq
    return ll, chisq

def log_probability(p):
    
    lp = log_prior(p)

    if not np.isfinite(lp):
        return -np.inf, log_likelihood(p)[1]
    return lp + log_likelihood(p)[0], log_likelihood(p)[1]

<h3> Use emcee to run the chains </h3>

In [ ]:
import emcee
from multiprocessing import Pool

In [ ]:
def run_emcee(initial_seed_galaxy_bias, initial_seed_ia_params):
    
    if config_specific['IA_model'] == 'NLA':
        n_dim = 3
    else:
        n_dim = 5

    aprox_bias = np.concatenate([initial_seed_galaxy_bias, initial_seed_ia_params])

    path_chains = '/nfs/pic.es/user/d/dnavarro/IATheory/data/chains/' # Save the chains
    filename = path_chains + "wgg_wgp_{}_{}_Mpc_h_emcee.h5".format(case, config_specific['min_scale_cut'])
    
    n_walkers = 16 #32
    n_steps = 100 #10000

    initial = aprox_bias + 0.1 * np.random.randn(n_walkers, n_dim)
    backend = emcee.backends.HDFBackend(filename)
    backend.reset(n_walkers, n_dim)
    
    # We'll track how the average autocorrelation time estimate changes
    index = 0
    autocorr = np.empty(n_steps)

    # This will be useful to testing convergence
    old_tau = np.inf
    
    # I run the chains
    with Pool() as pool:
        sampler = emcee.EnsembleSampler(
        n_walkers,
        n_dim,
        log_probability,
        moves=[(emcee.moves.DEMove(), 0.8), (emcee.moves.DESnookerMove(), 0.2)],
        pool=pool,
        backend=backend
        )
        # Now we'll sample for up to n_steps
        for sample in sampler.sample(initial, iterations=n_steps, progress=True):
            # Only check convergence every 100 steps
            if sampler.iteration % 100:
                continue

            # Compute the autocorrelation time so far
            # Using tol=0 means that we'll always get an estimate even
            # if it isn't trustworthy
            tau = sampler.get_autocorr_time(tol=0)
            autocorr[index] = np.mean(tau)
            index += 1

            # Check convergence
            converged = np.all(tau * 100 < sampler.iteration)
            converged &= np.all(np.abs(old_tau - tau) / tau < 0.01)
            if converged:
                break
            old_tau = tau
    
    return None

In [ ]:
initial_seed_galaxy_bias = [1.2, -0.4]
initial_seed_ia_params = [0.5, 1, 1.5]
run_emcee(initial_seed_galaxy_bias, initial_seed_ia_params)

<h3> Use Nautilus to run the chains </h3>

In [ ]:
from nautilus import Sampler, Prior

In [ ]:
config_specific['n_cores'] = 120

In [ ]:
def loglikelihood_fn(p_dict):
    # p_dict keys follow prior_obj keys order
    if config_specific['IA_model'] == 'NLA':
        param_names = ['b1', 'b2', 'a1']
    else:
        param_names = ['b1', 'b2', 'a1', 'a2', 'ad']

    p = np.array([p_dict[key] for key in param_names])
    logl, _ = log_likelihood(p)
    return logl

In [ ]:
def run_nautilus():

    prior_obj = Prior()  

    if config_specific['IA_model'] == 'NLA':
        prior_obj.add_parameter('b1', dist=(0, 3.0))
        prior_obj.add_parameter('b2', dist=(-5.0, 5.0))
        prior_obj.add_parameter('a1', dist=(0.0, 10.0))
    else:
        prior_obj.add_parameter('b1', dist=(0, 5.0))
        prior_obj.add_parameter('b2', dist=(-5.0, 5.0))
        prior_obj.add_parameter('a1', dist=(-50.0, 50.0))
        prior_obj.add_parameter('a2', dist=(-50.0, 50.0))
        prior_obj.add_parameter('ad', dist=(-50.0, 50.0))

    n_live = 5000

    output_path = '/disks/shear16/herle/models/IATheory/'

    filename = output_path + f"nautilus_chain_2.h5"
    
    sampler = Sampler(
        prior_obj,
        loglikelihood_fn,
        n_live=n_live,
        pool= config_specific['n_cores'] if config_specific['n_cores'] > 1 else None,
        filepath=filename,
        resume=False,
        n_networks=16,
    )
    sampler.run(verbose=True)
    points, log_w, log_l = sampler.posterior()
    weights = np.exp(log_w)
    logz = sampler.log_z
    output_file = (output_path + f"nautilus_chain_2.npz")

    np.savez(output_file,
            samples=points,
            logl=log_l,
            weights=weights,
            logz=logz)
    
    print(f"Saved results to {output_file}")

    return None
